<h1 align="center"><b>Homework Assignment 5 (100 points total)</b></h1>
<h3 align="center"><b>Assigned at the start of Module 12</b></h3>
<h3 align="center"><b>Due at the end of Module 14</b></h3><br>

---

# Q1 — Study and Summarize *"Attention Is All You Need"*💡
 
## 20 points total

### Objective

- Read and summarize the 2017 paper *"Attention Is All You Need"* (Vaswani et al., NeurIPS 2017), which introduced the Transformer architecture behind modern models such as GPT and BERT. 
- Understand how attention mechanisms replace recurrence, enable efficient parallel training, and transform the design of large-scale generative AI systems.

### What to Include in Your Report

Write a concise report of no more than three pages in your own words that addresses the following:

**A. Motivation** – What limitations of RNNs and CNNs did the Transformer overcome?  
**B. Architecture**
- Summarize the encoder-decoder structure, scaled dot-product attention, and positional encodings. 
- Include one labeled diagram illustrating the Transformer block.  

**C. Training & Efficiency** – Explain how self-attention enables full-sequence parallelization and captures long-range dependencies.  
**D. Results & Impact** – Highlight key translation results and discuss why this work reshaped generative modeling.  
**E. Reflection & Analogy** 
- Provide an example from your own field or area of expertise (e.g., biology, finance, public policy, engineering, etc.) where a Transformer-like attention process could enhance performance or understanding. 
- Map elements of your example to Transformer components (e.g., queries $\rightarrow$ questions, keys $\rightarrow$ evidence, values $\rightarrow$ information sources).

Cite the original paper and any additional references used. References do not count toward the page limit.


---

# Q2 — Mini-Transformer Encoder for Short Texts

## 40 points total

### Objective

Transformers learn relationships between words/tokens using self-attention instead of recurrence as in RNNs. In this problem, you will build and train a Mini-Transformer Encoder to classify short texts from the AG News dataset (4 topics: World, Sports, Business, Sci/Tech). This hands-on task demonstrates how self-attention converts raw word sequences into context-aware representations that capture dependencies between tokens and enable effective topic classification in modern language models.

### [15 points] Part A – Prepare Dataset


__Goal:__ Load, understand and preprocess the AG News dataset for Transformer-based text classification.

1. Use the AG News dataset. (Reference: https://huggingface.co/datasets/fancyzhx/ag_news)

    ``` python
    from datasets import load_dataset  
    dataset = load_dataset("ag_news")
    ```

2. Each sample consists of a short news headline and label (0–3). Confirm label mappings:  0 = World 1 = Sports 2 = Business 3 = Sci/Tech. Show a few examples.
3. Tokenize headlines with a pretrained tokenizer, e.g., `bert-base-uncased` (Reference: https://huggingface.co/docs/transformers/en/tokenizer_summary).
    - Pad or truncate to 24 tokens.
    - Include `[CLS]` at position 0 and `[SEP]` if the tokenizer does so by default.
    - Return ```input_ids``` and ```attention_mask``` (1 for real tokens, 0 for padding).
4. Create train/validation/test splits and Dataloaders.
    - From the provided train split, create 90/10 train/val. Keep the original test set.
    - Build PyTorch DataLoaders with `batch_size = 64` that yield: `input_ids: [B, 24]`, `attention_mask: [B, 24]`, `labels: [B]`

### [15 points] Part B – Implement and Train the Mini-Transformer Encoder


__Goal__: Build a small encoder-only Transformer and train it for 4-way topic classification.

Use PyTorch:
- Token Embedding: `nn.Embedding(vocab_size, d_model=64)` (use the tokenizer’s `vocab_size`).
- Positional Encoding: Add sinusoidal positional encodings to token embeddings.
- Encoder Block: One Pre-LN layer with
    - Multi-Head Self-Attention (2 heads, d_model = 64)
    - Feed-Forward Network (hidden = 128, activation = GELU)
    - Residual + LayerNorm around MHSA and FFN (Pre-LN).
    - Dropout=0.1 applied to attention probs and FFN activations.
    - Padding mask: use `attention_mask` to prevent attending to pads.

- Classifier: Take the contextualized [CLS] token embedding vector at index 0 $\rightarrow$ Linear(64 $\rightarrow$ 4)

Training: 3 epochs with AdamW (`lr = 2e-4`, `weight_decay = 0.01`). Print training and validation accuracy each epoch and final test accuracy.


### [10 points] Part C – Visualize and Reflect


__Goal__: Inspect attention and reason about complexity.

- Attention heatmap: Pick one correctly classified test headline. For one head in the encoder, plot the [24×24] attention weights (query on y-axis, key on x-axis). Label tokens along both axes (use the tokenizer to decode).
- Reflect what you observe:
    - Do higher attention weights align with key topic words?
    - Do higher attention weights land on topic-bearing words (e.g., team names, companies, countries)?
    - Why does self-attention cost as $O(n^2)$ in sequence length $n$?


---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Part A - Preparing the Dataset

In [ ]:
raw = load_dataset("ag_news")
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# Show examples + verify label mapping
for i in range(5):
    ex = raw["train"][i]
    print(f"[{label_map[ex['label']]}] {ex['text'][:120]}")

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 24

class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=24):
        self.texts, self.labels = list(texts), list(labels)
        self.tok, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], padding="max_length", truncation=True,
                       max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[i], dtype=torch.long),
        }

# 90/10 stratified split off of the original training set, test set stays as-is
tr_x, va_x, tr_y, va_y = train_test_split(
    raw["train"]["text"], raw["train"]["label"],
    test_size=0.1, random_state=42, stratify=raw["train"]["label"]
)

train_ds = AGNewsDataset(tr_x, tr_y, tok, MAX_LEN)
val_ds   = AGNewsDataset(va_x, va_y, tok, MAX_LEN)
test_ds  = AGNewsDataset(raw["test"]["text"], raw["test"]["label"], tok, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, num_workers=2)

# Sanity, checking shapes + that [CLS]=101 sits at position 0
b = next(iter(train_loader))
print(b["input_ids"].shape, b["attention_mask"].shape, b["label"].shape)
print("First token ids (should start with 101 = [CLS]):", b["input_ids"][:3, 0])

# Q3 — Hybrid AI: Adding Logic Rules to a Neural Model

## 40 points total

### Objective

Understand how symbolic rules can guide neural models toward more consistent and explainable predictions.

### [10 points] Part A – Define a Rule

Write one simple logic rule, e.g.:

- IF income < 20000 THEN loan_approval = False, or
- IF temperature > 38 THEN illness = “fever-related”


### [10 points] Part B — Build a Small Neural Model

1. Create a small synthetic dataset (10–20 samples).
2. Train a simple logistic regression or 1-layer MLP (you may use `sklearn` or `numpy`).
3. Report baseline accuracy.

### [10 points] Part C — Add a Rule Penalty

- Add a check inside training or evaluation:
- If the model violates your rule, add a small penalty (e.g., +1 to loss).
- Show how this affects model output or loss after 5–10 iterations.

### [10 points] Part D — Interpret

Describe:
- How the rule influenced the model’s predictions.
- Why hybrid systems (logic + learning) are more trustworthy.